# OSNet: контролируемый перебор улучшений — variant 14

Исходные bbox, метки, строки train и validation **не изменяем**. Полное обучение запускаешь ты. Результаты изолированы от MVP.

22 конфигурации → подтверждение на трёх seed → второй inner split → замороженный refit и outer report. См. [README](README.md). Интернет/новые веса не требуются. Внешний teacher, faithful RPTM и комбинации — отдельный следующий этап, не выдаём их за выполненные.


In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/osnet_ablation_suite.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из каталога проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
import pandas as pd
from dataclasses import asdict
from IPython.display import Markdown, display
from training.osnet_ablations import experiment_grid
from training.osnet_ablation_suite import Budget, prepare, run_suite

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO)
print('Torch:', torch.__version__)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK
Torch: 2.14.0


## 1. Настройки, зафиксированные до результатов

Полный план — до 36 уникальных обучений, максимум 61 200 optimizer updates. Для первичного короткого запуска можно сократить список до B0/N1/C1/P1/A1 **до первого запуска** и задать отдельный RUN_NAME. Это будет частичный screening, не полный перебор. Изменение бюджета тоже требует нового RUN_NAME. Имена checkpoint и история не перезаписывают другие runs.


In [2]:
RUN_NAME = 'suite_v1'
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
ALLOW_CPU_TRAINING = False  # Preflight допустим; многодневный CPU-перебор требует явного выбора.
BUDGET = Budget(max_steps=1700, evaluation_interval=200, warmup_steps=100)
SEEDS = (20260915, 20260916, 20260917)
VARIANT_NAMES = list(experiment_grid())  # Все 22, включая B0.
FINAL_EVALUATION = True  # False: только inner screening/confirmation/alternate, без outer/refit/export.

print('Device:', DEVICE)
print('Budget:', asdict(BUDGET))
display(pd.DataFrame([{'variant': name, 'hypothesis': experiment_grid()[name].description,
                      'batch': experiment_grid()[name].p * experiment_grid()[name].k}
                     for name in VARIANT_NAMES]))


Device: mps
Budget: {'max_steps': 1700, 'evaluation_interval': 200, 'warmup_steps': 100}


,variant,hypothesis,batch
0,B0_control,"Stock-start, исходный рецепт, фиксированный st...",32
1,N1_backbone_bn,Freeze backbone BN statistics; affine обучается,32
2,N2_all_bn,Freeze backbone и BNNeck statistics,32
3,C1_no_consistency,Без self-consistency,32
4,C2_ema_consistency,"EMA target, decay=0.99; не внешний teacher",32
5,P1_p16k4,"P16K4; matched updates, удвоенный exposure",64
6,P2_p32k2,P32K2; больше identity в batch,64
7,A1_no_lower_occlusion,Без lower-center occlusion,32
8,A2_weak_lower_occlusion,Lower-center occlusion p=0.1,32
9,A3_weak_erasing,RandomErasing p=0.1,32


## 2. Preflight: только чтение исходников и данных

Сверяем все изображения, bbox/метки, splits, initializer, frozen mask cache, evaluator и runtime. Manifest записывается только в variant_14/runs. Если данные отличаются, notebook останавливается, а не «чинит» их. Общий inner holdout отделён также от identity/кадров локального YOLO. Ни одна строка из outer train не удаляется.


In [3]:
context = prepare(RUN_NAME, BUDGET, SEEDS, VARIANT_NAMES, device=DEVICE)
print('Output:', context['output'])
print('Rows:', len(context['rows']))
print('Outer identities:', {k: len(v) for k, v in context['manifest']['outer'].items()})
print('Inner identities:', {n: {k: len(v) for k, v in split.items()}
                            for n, split in context['manifest']['inner'].items()})
print('Source/protocol fingerprint:', context['signature'])


Source integrity (no edits):   0%|          | 0/9556 [00:00<?, ?it/s]

Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_14_controlled_ablations/runs/suite_v1
Rows: 9556
Outer identities: {'train': 925, 'calibration': 307, 'validation': 309}
Inner identities: {'primary': {'train': 740, 'validation': 185}, 'alternate': {'train': 740, 'validation': 185}}
Source/protocol fingerprint: 20e698bdf01691483866025d3272d5dd4fbacaa331092bf12ee889d027322bb5


## 3. Последовательный запуск с возобновлением

Best checkpoint определяется только по raw inner mAP@10. B0 и два лучших изменения сравниваются на тех же трёх seed. Порог выбирается только на calibration после замораживания final weights. Outer validation не участвует ни в выборе варианта, ни в выборе шага. Второй split проверяет устойчивость, но не переопределяет победителя.

При прерывании: Restart Kernel → Run All с теми же настройками. Последний незавершённый блок до 200 steps повторится. Не запускай одну серию в двух kernel. Ошибки не пропускаются автоматически: checkpoint до ошибки сохраняется. Не редактируй код в процессе обучения.


In [4]:
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('Полный CPU-перебор отключён. Выбери CUDA/MPS или явно ALLOW_CPU_TRAINING=True.')
result = run_suite(context, final_evaluation=FINAL_EVALUATION)
print('Finished. MVP promoted:', result['promoted'])


primary/B0_control/20260915: 200/1700 | inner mAP=0.7779, best=0.7779 | elapsed 00:03:20 | ETA 00:24:58
primary/B0_control/20260915: 285/1700 | inner mAP=0.7864, best=0.7864 | elapsed 00:04:42 | ETA 00:23:22
primary/B0_control/20260915: 400/1700 | inner mAP=0.8048, best=0.8048 | elapsed 00:06:31 | ETA 00:21:11
primary/B0_control/20260915: 600/1700 | inner mAP=0.8141, best=0.8141 | elapsed 00:09:33 | ETA 00:17:30
primary/B0_control/20260915: 800/1700 | inner mAP=0.8064, best=0.8141 | elapsed 00:12:36 | ETA 00:14:10
primary/B0_control/20260915: 850/1700 | inner mAP=0.8056, best=0.8141 | elapsed 00:13:28 | ETA 00:13:28
primary/B0_control/20260915: 1000/1700 | inner mAP=0.8098, best=0.8141 | elapsed 00:15:48 | ETA 00:11:03
primary/B0_control/20260915: 1200/1700 | inner mAP=0.8039, best=0.8141 | elapsed 00:18:50 | ETA 00:07:51
primary/B0_control/20260915: 1400/1700 | inner mAP=0.8101, best=0.8141 | elapsed 00:21:53 | ETA 00:04:41
primary/B0_control/20260915: 1600/1700 | inner mAP=0.8161, be

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


primary/M2_xbm1024/20260915: 200/1700 | inner mAP=0.7590, best=0.7590 | elapsed 00:03:00 | ETA 00:22:30
primary/M2_xbm1024/20260915: 285/1700 | inner mAP=0.7710, best=0.7710 | elapsed 00:04:21 | ETA 00:21:38
primary/M2_xbm1024/20260915: 400/1700 | inner mAP=0.7767, best=0.7767 | elapsed 00:06:08 | ETA 00:19:56
primary/M2_xbm1024/20260915: 600/1700 | inner mAP=0.7995, best=0.7995 | elapsed 00:09:07 | ETA 00:16:44
primary/M2_xbm1024/20260915: 800/1700 | inner mAP=0.8011, best=0.8011 | elapsed 00:12:07 | ETA 00:13:38
primary/M2_xbm1024/20260915: 850/1700 | inner mAP=0.8063, best=0.8063 | elapsed 00:12:58 | ETA 00:12:58
primary/M2_xbm1024/20260915: 1000/1700 | inner mAP=0.7951, best=0.8063 | elapsed 00:15:14 | ETA 00:10:40
primary/M2_xbm1024/20260915: 1200/1700 | inner mAP=0.8080, best=0.8080 | elapsed 00:18:13 | ETA 00:07:35
primary/M2_xbm1024/20260915: 1400/1700 | inner mAP=0.8051, best=0.8080 | elapsed 00:21:13 | ETA 00:04:33
primary/M2_xbm1024/20260915: 1600/1700 | inner mAP=0.8115, be

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


Finished. MVP promoted: False


## 4. Результаты и ограничения

P1/P2 имеют двойной exposure при одинаковых updates; сравни дополнительно step 850 с B0 step 1700 в history.json. Masked условия — только дополнительные diagnostics с фиксированным original-calibration threshold. Основная validation всегда original.

Рост одного screening seed не доказывает улучшение. Смотри paired seeds, alternate split, mask regression, candidate score и conditional bootstrap. CPU forward timing из export.json — **не** полный A5000 extract benchmark. Никакой вариант автоматически не заменяет MVP.


In [5]:
display(Markdown((context['output'] / 'RESULTS.md').read_text(encoding='utf-8')))
print('Detailed artifacts:', context['output'])


# OSNet variant 14 — результаты

Исходные строки/bbox/validation не изменены. MVP не заменён.
Selection: primary inner raw mAP@10; outer используется только после freeze выбора.

## Screening (один seed, не доказательство)

| Вариант | inner mAP@10 | best step | samples seen |
|---|---:|---:|---:|
| B0_control | 0.81612 | 1600 | 54400 |
| N1_backbone_bn | 0.82254 | 1600 | 54400 |
| N2_all_bn | 0.78004 | 1600 | 54400 |
| C1_no_consistency | 0.82249 | 1600 | 54400 |
| C2_ema_consistency | 0.82279 | 600 | 54400 |
| P1_p16k4 | 0.82161 | 600 | 108800 |
| P2_p32k2 | 0.81167 | 285 | 108800 |
| A1_no_lower_occlusion | 0.82618 | 800 | 54400 |
| A2_weak_lower_occlusion | 0.82232 | 800 | 54400 |
| A3_weak_erasing | 0.82574 | 1700 | 54400 |
| M1_am_softmax | 0.79533 | 1200 | 54400 |
| M2_xbm1024 | 0.81277 | 1700 | 54400 |
| M3_triplet | 0.82978 | 1700 | 54400 |
| M4_circle | 0.74769 | 1200 | 54400 |
| R1_resolution256 | 0.82909 | 1200 | 54400 |
| R2_local128 | 0.81908 | 600 | 54400 |
| K1_color32 | 0.81084 | 800 | 54400 |
| T1_balanced_positives | 0.83126 | 1400 | 54400 |
| S1_mixed_masks | 0.81379 | 600 | 54400 |
| G1_gem | 0.81612 | 1600 | 54400 |
| L1_letterbox | 0.79191 | 1200 | 54400 |
| S2_mixstyle | 0.81612 | 1600 | 54400 |

## Confirmation: три seed

| Вариант | mean | sample std | final steps |
|---|---:|---:|---:|
| B0_control | 0.82093 | 0.00420 | 1600 |
| M3_triplet | 0.82626 | 0.00389 | 1700 |
| T1_balanced_positives | 0.83016 | 0.00245 | 1400 |

Замороженный выбор: **T1_balanced_positives**. Альтернативный split не меняет выбор.

## Alternate frame-grouped split

| Вариант | seed | inner mAP@10 |
|---|---:|---:|
| B0_control | 20260915 | 0.81555 |
| B0_control | 20260916 | 0.82261 |
| B0_control | 20260917 | 0.81564 |
| T1_balanced_positives | 20260915 | 0.82868 |
| T1_balanced_positives | 20260916 | 0.81774 |
| T1_balanced_positives | 20260917 | 0.82102 |

## Outer validation — только отчёт

| Вариант / input | raw mAP@10 | reranked mAP@10 | F1 | TNR | 0.7F1+0.3TNR |
|---|---:|---:|---:|---:|---:|
| MVP / frozen reference | 0.79016 | 0.81469 | 0.72861 | 0.79032 | 0.74712 |
| B0_control / original | 0.77769 | 0.79215 | 0.77477 | 0.59677 | 0.72137 |
| B0_control / masked_query | 0.75731 | 0.77608 | 0.76389 | 0.67742 | 0.73795 |
| B0_control / masked_gallery | 0.75470 | 0.77383 | 0.75751 | 0.64516 | 0.72380 |
| B0_control / masked_both | 0.73477 | 0.76054 | 0.74126 | 0.62903 | 0.70759 |
| T1_balanced_positives / original | 0.77271 | 0.78428 | 0.76372 | 0.80645 | 0.77654 |
| T1_balanced_positives / masked_query | 0.75214 | 0.77549 | 0.72277 | 0.82258 | 0.75271 |
| T1_balanced_positives / masked_gallery | 0.75557 | 0.76066 | 0.72683 | 0.77419 | 0.74104 |
| T1_balanced_positives / masked_both | 0.73207 | 0.73961 | 0.71744 | 0.77419 | 0.73447 |

Validation уже использовалась в исследованиях: это development, не независимый финальный тест.
MVP reference взят из проверенного baseline_metrics.json, не из нового обучения; hashes зафиксированы.
Mask-условия — дополнительные diagnostics; основная validation остаётся original.
CPU forward timing не заменяет официальный A5000 extract benchmark. Автопродвижения нет.


Detailed artifacts: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_14_controlled_ablations/runs/suite_v1
